In [1]:
from pyi18next.utility import get_plural_func
from core.settings import get_settings
from services.multilingual_manager import MultilingualManager
from services.encoder_factory import EncoderFactory
from services.calibrator_factory import CalibratorFactory
from services.model_registry import ModelRegistry
from pyi18next.backends.fs import Backend
from pyi18next.i18next import I18next
from adaptation.graph_builder import LocalizationGraphBuilder
from adaptation.utils import traverse_namespaces


In [4]:
settings = get_settings()
languages = settings.languages
print(settings)


languages={'es'} bert={'es': 'dccuchile/bert-base-spanish-wwm-cased'} sbert={'es': 'hiiamsid/sentence_similarity_spanish_es'} word2vec={'es': 'C:/Users/malos/Documents/GitHub/JustShare/server/models/spanish_word2vec/word2vec.bin'} spacy={'es': 'es_core_news_sm'} siamese_lstm={'es': 'C:/Users/malos/Documents/GitHub/JustShare/server/models/lstm_mean_cosine_noaug'} allow_origins=['http://localhost:8080', 'http://127.0.0.1:8080'] host='0.0.0.0' port=8000


In [5]:
base_dir = "./adaptation/localization/final"


In [6]:
namespaces = traverse_namespaces(base_dir, languages)
print(namespaces)


['scene6/routeA/scene6BedroomRouteA2', 'transitions', 'menus/loginScene', 'names', 'deviceInfo', 'scene4/scene4Garage', 'generalDialogs', 'scene4/scene4Frontyard', 'scene6/scene6Bedroom', 'scene6/routeA/scene6BedroomRouteA1', 'scene5/scene5Bedroom', 'scene3/scene3Break', 'scene6/routeB/scene6LunchRouteB', 'computer/socialMediaScreen', 'scene1/scene1Bedroom1', 'scene1/scene1Classroom', 'scene3/scene3Bedroom', 'scene6/routeB/scene6BedroomRouteB', 'scene5/scene5Livingroom', 'scene2/scene2Bedroom', 'computer/captions', 'computer/usernames', 'scene4/scene4Bedroom', 'scene6/routeA/scene6PortalRouteA', 'scene1/scene1Lunch2', 'scene7/scene7Bedroom', 'menus/creditsScene', 'scene6/routeA/scene6EndingRouteA', 'scene6/routeB/scene6PoliceStationRouteB', 'scene1/scene1Break', 'scene1/scene1Bedroom2', 'menus/titleScene', 'scene6/routeB/scene6EndingRouteB', 'computer/loginScreen', 'scene4/scene4Backyard', 'scene6/routeA/scene6LunchRouteA', 'scene1/scene1Lunch1', 'scene2/scene2Break', 'scene6/scene6Liv

In [5]:
backend = Backend(name_mapping=lambda lng, ns: f"{base_dir}/{lng}/{ns}.json")

i18n = I18next(
	backend=backend,
	lng=list(languages),
	ns=namespaces,
)



In [6]:
# rules = "one: n is 1; other:"
rules = {
	"one": "n is 1",
	"other": ""
}

plural_func = get_plural_func(rules)

print(plural_func(1))
print(plural_func(3))


one
other


In [7]:
base_dir = "./faiss_data"

model_registry = ModelRegistry(languages)
model_registry.build_tranformer("sbert")
model_registry.resolve_all()
encoder_factory = EncoderFactory(model_registry)
calibrator_factory = CalibratorFactory(model_registry)
multilingual = MultilingualManager(encoder_factory, calibrator_factory, base_dir)
model_types = model_registry.active_model_types()


2026-05-19 23:10:24.558 | DEBUG    | services.model_registry:_create_loader:60 - Registering sbert loader for 'es'
2026-05-19 23:10:24.561 | DEBUG    | services.lazy_loader:model:16 - Loading sbert for 'es'...


Using device: cuda


2026-05-19 23:10:27.507 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded sbert for 'es'


In [9]:
builder = LocalizationGraphBuilder(
    i18n=i18n,
    languages=languages,
    multilingual=multilingual,
    model_registry=model_registry,
    base_dir="./adaptation/localization/structure",
)

builder.run()


scene1/scene1Classroom


2026-05-19 21:14:54.825 | DEBUG    | controllers.retrievers.faiss:_fit:93 - Indexed 31 vectors
2026-05-19 21:14:54.855 | DEBUG    | services.node_engine:save_node:42 - Saving FAISS node | model=sbert | language=es | node=scene1Classroom_part2_thanks2


Total visited nodes: 670


In [10]:
multilingual = MultilingualManager(encoder_factory, calibrator_factory, base_dir)
test_engine = multilingual.get_node_engine("es", "sbert")

print(test_engine.retrievers)

test_engine.load_all()

print(test_engine.retrievers)


2026-05-19 21:14:54.880 | DEBUG    | services.node_engine:load_node:54 - Loading FAISS node | model=sbert | language=es | node=scene1Classroom_part2_thanks2
2026-05-19 21:14:54.885 | SUCCESS  | services.node_engine:load_node:68 - Loaded node successfully.


{}
{'scene1Classroom_part2_thanks2': <controllers.retrievers.faiss.FaissRetriever object at 0x000001C8D2CCB4D0>}


In [11]:
retriever = test_engine.get_retriever("scene1Classroom_part2_thanks2")

retriever.search("Hola", 3)


(array([29, 17, 23], dtype=int32),
 array([1.        , 0.5254722 , 0.51313424], dtype=float32),
 array(['Hola', 'Holaaa, soy [UNK] que ta', 'Hola! Soy [UNK]'],
       dtype=object))